In [ ]:
import os
import requests
from typing import Dict, Optional
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from vertexai.preview.reasoning_engines import AdkApp

In [ ]:
AGENT_MODEL = "gemini-2.5-flash"

In [ ]:
basic_agent = Agent(
    name="basicAgent",
    model=AGENT_MODEL,
    description="A friendly agent",
    instruction="greet the user with a hello in Spanish and then list the coordinates of West Lafayette, IN."
)

app = AdkApp(
    agent=basic_agent
)

user_id = "test-user-01"

message = f"Hello there!"

for event in app.stream_query(user_id=user_id, message=message):
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            if "text" in part:
                print(part["text"])

¡Hola!

The coordinates of West Lafayette, Indiana are approximately:
Latitude: 40.4259° N
Longitude: 86.9081° W


In [ ]:
import requests
from typing import Dict, Any, Optional

def get_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
    """Retrieves the weather forecast from the National Weather Service (NWS) API.

    This function takes a latitude and longitude, first determines the
    appropriate NWS grid endpoint for that location, and then fetches the
    detailed weather forecast.

    Args:
        latitude: The latitude of the location (e.g., 38.8951).
        longitude: The longitude of the location (e.g., -77.0364).

    Returns:
        A dictionary containing the forecast properties, which includes a
        list of forecast periods. Returns None if the forecast could
        not be retrieved due to an API error or network issue.
    """
    # The NWS API requires a unique User-Agent header.
    # Replace "(My Weather App, myemail@example.com)" with your app's details.
    headers = {
        'User-Agent': '(My Weather App, myemail@example.com)'
    }

    # Step 1: Get the gridpoint metadata URL from the initial coordinates.
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"

    try:
        # First API call to get the specific forecast URL
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        # Extract the URL for the actual forecast from the response
        forecast_url = points_response.json()['properties']['forecast']

        # Step 2: Get the actual forecast data from the forecast URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Return the 'properties' key which contains the forecast data
        return forecast_response.json()['properties']

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while communicating with the NWS API: {e}")
        return None
    except (KeyError, ValueError) as e:
        # Handle cases where the JSON response is not as expected
        print(f"Error parsing the API response: {e}")
        return None

In [ ]:
!pip install googlemaps -q

In [ ]:
import googlemaps
from typing import Optional, Tuple

def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """Converts a physical address to latitude and longitude using Google's Geocoding API.

    Args:
        api_key: Your Google Maps API key with the Geocoding API enabled.
        address: The human-readable address or place name to geocode.

    Returns:
        A tuple containing the latitude and longitude, or None if an error occurs.
    """
    # Initialize the client with your API key
    gmaps = googlemaps.Client(key="")

    try:
        # Make the API call to geocode the address
        geocode_result = gmaps.geocode(location)

        if not geocode_result:
            print(f"Warning: No results found for address: '{location}'")
            return None

        location = geocode_result[0]['geometry']['location']
        latitude = location['lat']
        longitude = location['lng']

        return (latitude, longitude)

    except googlemaps.exceptions.ApiError as e:
        print(f"An API error occurred: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

print(get_lat_lon("New York, NY"))

(40.7127753, -74.0059728)


In [ ]:
weather_agent = Agent(
    name="weather_agent_v1",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Provides weather information for specific cities.",
    instruction="""You are a helpful weather assistant. When the user asks for the weather in a city,
     use get_lat_lon to convert that city into latitude and longitude and then call the get_weather_forecat tool with those cooridnates.""",
    tools=[get_lat_lon, get_weather_forecast], # Pass the function directly
)


app = AdkApp(
    agent=weather_agent
)

user_id = "test-user-02"

test_cities = [
    "New York, NY",
    "Los Angeles, CA",
    "Chicago, IL",
]

for city in test_cities:
    print(f"--- Fetching Weather for {city} ---")
    message = f"What is the current weather forecast for {city}?"

    for event in app.stream_query(user_id=user_id, message=message):
        if "content" in event and "parts" in event["content"]:
            for part in event["content"]["parts"]:
                if "text" in part:
                    print(part["text"])
    print("\n")

--- Fetching Weather for New York, NY ---


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


The current weather forecast for New York, NY:

Today: A chance of snow before 9am, then rain and snow between 9am and 11am, then rain and snow. Cloudy. High near 41, with temperatures falling to around 38 in the afternoon. East wind around 8 mph. Chance of precipitation is 100%. New snow accumulation of less than half an inch possible. (Short forecast: Chance Light Snow then Rain And Snow)

Tonight: Rain before 4am. Cloudy. Low around 38, with temperatures rising to around 41 overnight. Southwest wind 6 to 10 mph. Chance of precipitation is 90%. New rainfall amounts between a half and three quarters of an inch possible. (Short forecast: Light Rain)


--- Fetching Weather for Los Angeles, CA ---
The current weather forecast for Los Angeles, CA is: Today, there will be patchy fog then it will be mostly sunny, with a high of 73°F. The wind will be from the south southwest at 0 to 10 mph.


--- Fetching Weather for Chicago, IL ---
The current weather forecast for Chicago, IL is mostly clo

In [ ]:
import vertexai

vertexai.init(
    project="qwiklabs-gcp-00-819951dd6b01",
    location="us-central1",
    staging_bucket="gs://adk-capstone-east"
)

from vertexai.preview.reasoning_engines import AdkApp
app = AdkApp(agent=weather_agent)
for event in app.stream_query(
user_id="USER_ID",
message="I'm a resident of MD and heard about an impending blizzard. Is this true?",
):
  print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-6379b9e7-fef7-47e3-b417-0514e9ea75e3', 'args': {'location': 'MD'}, 'name': 'get_lat_lon'}, 'thought_signature': 'CvcJAY89a1_Uf7HMjQ_x09Off4EK7kfFf6tNw_KF4pW7pxFgSjMculbFOah7p0lOJ5T4j3NzeMzRGJISpApHHqqc8gX-f_tBN2xSR4s6mnPDSjStPaXQ5aXEPHvvVUUBRnhloLoMa3Gl4a1an2gRVIQ5-bydG27Eb-Z_xtLsiRHpIAbK7sVF0aXP6lnPqniLe2Zj34FduO2-E3Gc76bT_qdVx6WhY2mVaS3lG-z__sKz84z3_LkquMVASFjrI9QDPwOW6F0-mU3v8exHYx_xbW1Mq2wXDLvBgmeMKS_uWxRC4xoIlIz2LGaUusZ3aPm_4ADhrmeTGbC6BdpxLMmOe9Bec1BRXE6xvrWr3cWteOHZ8Poa2vYv39iHZnfrqQOLC1K0eda1ccQ6cOYCbHQLLDyPx4gCCPb-CxCQduS2xmq9alGQ-maZPeCmHnaDHQcZhrvMz9-p6Qga1vTjRevIUhb8r8Bj0vQ3HNR_-ZZJXj2WCRsnMwsemTQA5pCiYCZNQQUYb6-k78LhzqobWd-632gVdAxIq4Q4mlhFp69gi9UKBA7knFEsYXqSox_0UNK9QsRGWsaQdMsRb7BqWXV5qt_5qms74mjt3jYOgol7Itx5huR_xLNC3aFEIqnxfY0zHtZO-MRWT7kp__7N_kV1JZOv-YF5OP_RAB-6sBXf9JlAbTapyKRfcx2AzA1bCM3ZXXQvZN8MXzMVXpN6vCs41FzMskJQuQ7Eir-nRS_9gqyFuWWLqvXfUT2UVwbDPhhSQeYSzCx01bxEeNRePaIXZcZfRcgj9